In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml kailash-align

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# M6 Ollama bootstrap — local LLMs, no API keys
# On Colab: installs + starts Ollama and pulls this lesson's models.
# Locally: verifies the daemon is running and models are pulled.
# ══════════════════════════════════════════════════════════════════
_LESSON_MODELS = ['qwen2.5:0.5b', 'llama3.2:3b']
import sys, os, time, shutil, subprocess
import httpx

_OLLAMA_BASE_URL = os.environ.get('OLLAMA_BASE_URL', 'http://localhost:11434')
_IN_COLAB = 'google.colab' in sys.modules

def _daemon_up(timeout_s=1.0):
    try:
        r = httpx.get(f'{_OLLAMA_BASE_URL}/api/tags', timeout=timeout_s)
        r.raise_for_status()
        return r.json()
    except Exception:
        return None

if _IN_COLAB and shutil.which('ollama') is None:
    print('[ollama] installing binary …')
    subprocess.run(
        'curl -fsSL https://ollama.com/install.sh | sh',
        shell=True, check=True,
    )

if _IN_COLAB and _daemon_up() is None:
    print('[ollama] starting daemon …')
    subprocess.Popen(
        'nohup ollama serve > /tmp/ollama.log 2>&1 &',
        shell=True,
    )
    deadline = time.monotonic() + 30
    while time.monotonic() < deadline and _daemon_up() is None:
        time.sleep(0.5)

_tags = _daemon_up(timeout_s=2.0)
if _tags is None:
    raise RuntimeError(
        f'Ollama daemon not reachable at {_OLLAMA_BASE_URL}. '
        'On Colab: re-run this cell. Locally: run `ollama serve`.'
    )

_have = {m.get('name','').split(':',1)[0] for m in _tags.get('models', [])}
for _model in _LESSON_MODELS:
    _fam = _model.split(':', 1)[0]
    if _fam in _have:
        continue
    if _IN_COLAB:
        print(f'[ollama] pulling {_model} (one-time) …')
        subprocess.run(['ollama', 'pull', _model], check=True)
    else:
        raise RuntimeError(
            f'Model {_model!r} not pulled. Run: ollama pull {_model}'
        )

print(
    '✓ Ollama ready at', _OLLAMA_BASE_URL,
    '— models:', _LESSON_MODELS,
)



In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp06
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


def create_visualizer():
    """Return a ModelVisualizer with the P2 experimental notice acknowledged.

    kailash-ml 2.2.2 emits ExperimentalWarning (a UserWarning) at
    ModelVisualizer construction; exercises run under warnings-as-errors,
    so the notice is acknowledged narrowly here — the ONE construction site
    every course file should use.
    """
    import warnings as _warnings

    from kailash_ml import ModelVisualizer
    from kailash_ml._decorators import ExperimentalWarning as _EW

    with _warnings.catch_warnings():
        _warnings.simplefilter("ignore", _EW)
        return ModelVisualizer()

# ── SDK-internal deprecation nag filter (strict-gate hygiene) ────────────
# torch 2.12's ONNX exporter warns that dynamic_axes is "not recommended"
# under dynamo=True — a deprecation notice inside OnnxBridge's internals,
# not a correctness signal, and fatal under warnings-as-errors. Acknowledged
# here (kailash_helpers is imported by every shared.mlfpNN package, so every
# technique file gets it); message-matched, everything else still errors.
import warnings as _warnings

_warnings.filterwarnings(
    "ignore", message=r".*dynamic_axes.*not recommended.*", category=UserWarning
)
# matplotlib's Agg backend (forced in headless/script runs) nags
# "FigureCanvasAgg is non-interactive, and thus cannot be shown" on every
# plt.show() — a backend notice, not a correctness signal.
_warnings.filterwarnings(
    "ignore", message=r".*non-interactive.*cannot be shown.*", category=UserWarning
)

def hdb_storey_range_expr(column: str = "storey_range"):
    """Polars expression normalising HDB storey_range letter-O typos.

    The raw HDB resale file has letter-O typos ("O4 TO 06", "1O TO 12",
    "28 TO 3O"); read a letter O next to a digit as zero, leaving "TO"
    alone. Single home for the rule (P2): M2 ex_8 and M3 ex_5/07 both
    normalise at the use-site.
    """
    import polars as pl

    return (
        pl.col(column)
        .str.replace_all(r"\bO(\d)", "0${1}")
        .str.replace_all(r"(\d)O\b", "${1}0")
    )



# ── shared/mlfp06/ex_2.py ──
"""
Shared infrastructure for Exercise 2 — LLM Fine-Tuning.

Contains: SFT dataset loader (IMDB -> instruction pairs), LoRA/adapter
parameter-counting helpers, kailash-align config factory, base
hyperparameters, and output directory. Technique-specific classes
(LoRALayer, AdapterLayer, etc.) live in the per-technique files.
"""

import os
from pathlib import Path

import polars as pl
import torch


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()
torch.manual_seed(42)
device = get_device()

# Output directory for all visualisation and training artifacts
REPO_ROOT = Path.cwd()
OUTPUT_DIR = REPO_ROOT / "outputs" / "ex2_finetuning"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ════════════════════════════════════════════════════════════════════════
# SHARED HYPERPARAMETERS
# ════════════════════════════════════════════════════════════════════════

D_MODEL = 512  # synthetic demo hidden dim for from-scratch LoRA / adapter
LORA_RANK = 8  # default LoRA rank for the demos
LORA_ALPHA = 16.0  # typical alpha = 2 * rank
ADAPTER_BOTTLENECK = 64  # default adapter bottleneck

# ════════════════════════════════════════════════════════════════════════
# SFT DATA LOADING — IMDB instruction-response pairs
# ════════════════════════════════════════════════════════════════════════

DATA_DIR = REPO_ROOT / "data" / "mlfp06" / "imdb"
DATA_DIR.mkdir(parents=True, exist_ok=True)
CACHE_FILE = DATA_DIR / "imdb_sft_2k.parquet"


def load_imdb_sft(
    n_rows: int = 2000, train_frac: float = 0.9
) -> tuple[pl.DataFrame, pl.DataFrame, pl.DataFrame]:
    """Load IMDB movie reviews and reformat as SFT instruction pairs.

    Returns:
        (full, train, eval) polars DataFrames with columns:
            instruction, response, text, label
    """
    if CACHE_FILE.exists():
        print(f"Loading cached IMDB SFT pairs from {CACHE_FILE}")
        sft_data = pl.read_parquet(CACHE_FILE)
    else:
        print("Downloading stanfordnlp/imdb from HuggingFace (first run)...")
        from datasets import load_dataset

        ds = load_dataset("stanfordnlp/imdb", split="train")
        ds = ds.shuffle(seed=42).select(range(min(n_rows, len(ds))))

        label_names = {0: "negative", 1: "positive"}
        rows = []
        for row in ds:
            review = row["text"][:1500]
            sentiment = label_names[row["label"]]
            rows.append(
                {
                    "instruction": (
                        "Classify the sentiment of the following movie review as "
                        "either 'positive' or 'negative', then briefly justify "
                        f"your answer.\n\nReview: {review}"
                    ),
                    "response": (
                        f"Sentiment: {sentiment}. The reviewer expresses a clearly "
                        f"{sentiment} reaction to the film."
                    ),
                    "text": review,
                    "label": sentiment,
                }
            )
        sft_data = pl.DataFrame(rows)
        sft_data.write_parquet(CACHE_FILE)
        print(f"Cached {sft_data.height} SFT pairs to {CACHE_FILE}")

    n_train = int(sft_data.height * train_frac)
    train_data = sft_data[:n_train]
    eval_data = sft_data[n_train:]
    print(
        f"IMDB SFT loaded: {sft_data.height} pairs "
        f"({train_data.height} train / {eval_data.height} eval)"
    )
    return sft_data, train_data, eval_data


# ════════════════════════════════════════════════════════════════════════
# PARAMETER COUNTING HELPERS
# ════════════════════════════════════════════════════════════════════════


def count_lora_params(
    d_model: int, rank: int, num_modules: int = 1, num_layers: int = 1
) -> int:
    """LoRA trainable params = 2 * d * r per target module per layer.

    For d=512, r=8, modules=1, layers=1 -> 8,192 params.
    """
    return (d_model * rank + rank * d_model) * num_modules * num_layers


def count_adapter_params(d_model: int, bottleneck: int, num_layers: int = 1) -> int:
    """Adapter params per layer: down(d*b+b) + up(b*d+d) + layernorm(2*d).

    Matches the AdapterLayer in 02_adapter_from_scratch.py.
    """
    return (
        d_model * bottleneck + bottleneck + bottleneck * d_model + d_model + 2 * d_model
    ) * num_layers


def full_finetune_params(
    d_model: int, num_modules: int = 1, num_layers: int = 1
) -> int:
    """Full fine-tuning baseline: d^2 per module per layer."""
    return d_model * d_model * num_modules * num_layers


# ════════════════════════════════════════════════════════════════════════
# KAILASH-ALIGN CONFIG FACTORY
# ════════════════════════════════════════════════════════════════════════


def get_base_model_name() -> str:
    """Return the HuggingFace repo id of the SFT base model.

    Resolution: ``SFT_BASE_MODEL`` env var → ``Qwen/Qwen2.5-0.5B-Instruct``.
    Note: Align/TRL training needs a HuggingFace repo id, NOT an Ollama tag —
    the Ollama-side ``OLLAMA_FT_BASE_MODEL`` is for inference of the
    fine-tuned model after GGUF export, not for training.
    """
    return os.environ.get("SFT_BASE_MODEL") or "Qwen/Qwen2.5-0.5B-Instruct"


def build_sft_config(
    base_model: str | None = None,
    lora_r: int = 16,
    lora_alpha: int = 32,
    num_epochs: int = 3,
    output_subdir: str = "sft_output",
):
    """Build a kailash-align AlignmentConfig for SFT+LoRA training.

    Imported lazily so technique files that do not call SFT are not
    forced to install the align extra at import time.

    kailash-align 0.6.0+ uses a composed AlignmentConfig: top-level
    method + base_model_id + experiment_dir, plus required LoRAConfig
    + SFTConfig + DPOConfig sub-configs. The DPOConfig is unused for
    SFT but the constructor still expects it (default values are fine).
    """
    from kailash_align import AlignmentConfig, DPOConfig, LoRAConfig, SFTConfig

    return AlignmentConfig(
        method="sft",
        base_model_id=base_model or get_base_model_name(),
        lora=LoRAConfig(
            rank=lora_r,
            alpha=lora_alpha,
            target_modules=("q_proj", "v_proj"),
            dropout=0.05,
        ),
        sft=SFTConfig(
            num_train_epochs=num_epochs,
            per_device_train_batch_size=4,
            gradient_accumulation_steps=4,
            learning_rate=2e-4,
            warmup_ratio=0.1,
            max_seq_length=512,
        ),
        dpo=DPOConfig(),
        experiment_dir=str(OUTPUT_DIR / output_subdir),
    )




# ════════════════════════════════════════════════════════════════════════
# MLFP06 — Exercise 2.1: LoRA From Scratch (Low-Rank Adaptation)
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Load an SFT (supervised fine-tuning) dataset of instruction-response pairs
#   - Implement LoRA as an nn.Module: W_new = W + (A @ B) * (alpha / r)
#   - Understand the low-rank intuition: only r directions are updated
#   - Visualise the parameter reduction curve across ranks r = 2..128
#   - Apply LoRA rank selection to a Singapore legal-document assistant
#
# PREREQUISITES: Exercise 1 (LLM fundamentals); M4.3 (SVD, low-rank factorisation)
# ESTIMATED TIME: ~35 min
#
# TASKS:
#   1. Load IMDB SFT dataset (instruction/response pairs)
#   2. THEORY: low-rank factorisation of the weight delta
#   3. BUILD: LoRALayer + LoRALinear (Kaiming A, zero B)
#   4. TRAIN: no-op — LoRA starts as identity, we verify parameter counts
#   5. VISUALISE: params vs rank sweep (log-scale plot saved to outputs/)
#   6. APPLY: Singapore law firm — rank selection cost/benefit
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import math

import matplotlib.pyplot as plt
import torch
import torch.nn as nn



## TASK 1 — Load SFT dataset


In [ ]:
print("\n" + "=" * 70)
print("TASK 1: Load SFT Dataset — IMDB instruction/response pairs")
print("=" * 70)

sft_data, train_data, eval_data = load_imdb_sft()
print(f"Sample instruction:\n  {sft_data['instruction'][0][:200]}...")
print(f"Sample response:\n  {sft_data['response'][0]}")



### Checkpoint 1


In [ ]:
assert sft_data.height > 0, "Task 1: SFT dataset should not be empty"
assert "instruction" in sft_data.columns, "Dataset needs 'instruction' column"
assert "response" in sft_data.columns, "Dataset needs 'response' column"
print(f"✓ Checkpoint 1 passed — {sft_data.height} SFT pairs loaded\n")

# INTERPRETATION: SFT data = (instruction, response) pairs.  The model learns
# to follow instructions by maximising P(response | instruction).  Quality
# matters more than quantity: 500 high-quality domain-specific pairs can
# outperform 10,000 generic pairs.



## THEORY — Low-Rank Adaptation (LoRA)


In [ ]:
# Full fine-tuning: W_new = W + delta_W, where delta_W is d x d.
#   That's d^2 trainable parameters per weight matrix. For d=4096 (a 7B
#   model's hidden size), that's 16.7M params PER MATRIX, and a 7B model
#   has hundreds of such matrices.
#
# LoRA insight (Hu et al., 2021): the UPDATE delta_W is low-rank.
#   Factorise it as delta_W = A @ B where A is d x r and B is r x d.
#   With r << d (typically r=8), trainable params drop from d^2 to 2*d*r.
#   For d=4096, r=8:  16.7M -> 65.5K params. A 255x reduction.
#
# Why it works: fine-tuning rarely needs the full expressiveness of
# d^2 directions. A handful of task-specific "directions" in weight
# space is enough.  LoRA IS low-rank factorisation (M4.3 SVD) applied
# to the weight UPDATE, not the weight itself.
#
# Initialisation trick: A ~ Kaiming, B = 0.  At init, A @ B = 0, so
# W_new = W (no change to the pretrained behaviour).  Training then
# gradually learns the delta.
#
# Analogy: think of the pretrained model as a piano.  Full fine-tuning
# rebuilds the whole instrument. LoRA is a capo — a small, cheap, and
# removable device that transposes the entire instrument into a new
# key without touching the strings.



## TASK 2 — BUILD: LoRALayer and LoRALinear


Computes: output = (x @ A @ B) * (alpha / r)

    Args:
        in_features:  Input dimension (d_model).
        out_features: Output dimension (d_model for self-attention projections).
        rank:         LoRA rank r.  Lower = fewer params, less expressive.
        alpha:        Scaling factor.  Typical: alpha = 2 * rank.
        dropout:      Dropout on the LoRA path for regularisation.


Ensures A @ B = 0 at init, so W_new = W (no change at start).


Computes: y = W_frozen @ x + bias + LoRALayer(x)


In [ ]:
class LoRALayer(nn.Module):

    def __init__(
        self,
        in_features: int,
        out_features: int,
        rank: int = 8,
        alpha: float = 16.0,
        dropout: float = 0.0,
    ):
        super().__init__()
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank

        # A: d_in x r  (Kaiming init for stable gradients)
        self.lora_A = nn.Parameter(torch.empty(in_features, rank))
        # B: r x d_out (zero init so LoRA starts as identity)
        self.lora_B = nn.Parameter(torch.zeros(rank, out_features))

        self.dropout = nn.Dropout(p=dropout) if dropout > 0 else nn.Identity()
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        dropped = self.dropout(x)
        lora_out = dropped @ self.lora_A @ self.lora_B
        return lora_out * self.scaling


class LoRALinear(nn.Module):

    def __init__(
        self,
        pretrained_linear: nn.Linear,
        rank: int = 8,
        alpha: float = 16.0,
        dropout: float = 0.0,
    ):
        super().__init__()
        self.linear = pretrained_linear
        # Freeze the pretrained weights
        for param in self.linear.parameters():
            param.requires_grad = False

        self.lora = LoRALayer(
            in_features=pretrained_linear.in_features,
            out_features=pretrained_linear.out_features,
            rank=rank,
            alpha=alpha,
            dropout=dropout,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear(x) + self.lora(x)



## TASK 3 — TRAIN: verify LoRA identity-at-init + parameter count


In [ ]:
# LoRA's training loop runs inside kailash-align's AlignmentPipeline
# (see 06_sft_alignment_pipeline.py).  Here we verify that the
# mathematical structure behaves as claimed: B=0 means zero delta at
# init, and the trainable count matches 2*d*r exactly.

print("=" * 70)
print("TASK 3: Verify LoRA structure on a synthetic d=512 linear")
print("=" * 70)

pretrained = nn.Linear(D_MODEL, D_MODEL).to(device)
lora_linear = LoRALinear(pretrained, rank=LORA_RANK, alpha=LORA_ALPHA).to(device)

frozen_params = sum(p.numel() for p in lora_linear.linear.parameters())
trainable_params = sum(
    p.numel() for p in lora_linear.lora.parameters() if p.requires_grad
)

# Identity-at-init check: W_new(x) should equal W(x) before any training
x_test = torch.randn(2, 10, D_MODEL, device=device)
with torch.no_grad():
    y_base = lora_linear.linear(x_test)
    y_lora = lora_linear(x_test)
identity_gap = (y_base - y_lora).abs().max().item()

print(f"LoRA layer created: d={D_MODEL}, r={LORA_RANK}")
print(f"  Frozen params:    {frozen_params:,}")
print(f"  Trainable params: {trainable_params:,}")
print(f"  Ratio:            {trainable_params / frozen_params:.4%} of frozen")
print(f"  Identity gap at init: {identity_gap:.2e} (should be ~0)")



### Checkpoint 2


In [ ]:
assert isinstance(lora_linear, LoRALinear), "Task 3: LoRALinear should be created"
assert (
    trainable_params == 2 * D_MODEL * LORA_RANK
), f"LoRA params should be 2*d*r = {2 * D_MODEL * LORA_RANK}"
assert identity_gap < 1e-5, "LoRA should start as identity (B=0)"
print("✓ Checkpoint 2 passed — LoRA structure and identity-init verified\n")

# INTERPRETATION: 2*d*r trainable params vs d^2 frozen.  For d=512, r=8:
# 8,192 vs 262,144 = 3.1%.  The B=0 initialisation means LoRA starts as
# identity, then gradually learns the domain-specific delta during training.



## TASK 4 — VISUALISE: parameter reduction curve across ranks


In [ ]:
print("=" * 70)
print("TASK 4: Visualise parameter reduction across ranks")
print("=" * 70)

# TinyLlama-scale analysis
hidden_dim = 2048
num_layers = 22
num_target_modules = 2  # q_proj, v_proj
full_total = full_finetune_params(hidden_dim, num_target_modules, num_layers)

ranks = [2, 4, 8, 16, 32, 64, 128]
lora_totals = [
    count_lora_params(hidden_dim, r, num_target_modules, num_layers) for r in ranks
]
pct_of_full = [p / full_total * 100 for p in lora_totals]

print(f"Model: TinyLlama-scale (d={hidden_dim}, layers={num_layers})")
print(f"Full fine-tuning params: {full_total:,}")
print(f"\n  rank    params         % of full    reduction")
for r, p, pct in zip(ranks, lora_totals, pct_of_full):
    print(f"  r={r:<5} {p:>12,}   {pct:>7.2f}%    {full_total / p:>6.1f}x")

# Plot: parameter count vs rank (log-log)
fig, ax = plt.subplots(1, 1, figsize=(9, 5))
ax.plot(ranks, lora_totals, "o-", color="steelblue", linewidth=2, label="LoRA")
ax.axhline(
    full_total, color="crimson", linestyle="--", label=f"Full FT = {full_total:,}"
)
ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.set_xlabel("LoRA rank (r)")
ax.set_ylabel("Trainable parameters")
ax.set_title("LoRA parameter reduction — TinyLlama-scale model", fontweight="bold")
for r, p, pct in zip(ranks, lora_totals, pct_of_full):
    ax.annotate(
        f"{pct:.1f}%", (r, p), textcoords="offset points", xytext=(5, 5), fontsize=8
    )
ax.legend()
ax.grid(True, alpha=0.3, which="both")
plt.tight_layout()
fname = OUTPUT_DIR / "ex2_lora_rank_sweep.png"
plt.savefig(fname, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"\n  Saved: {fname}")



### Checkpoint 3


In [ ]:
assert all(p < full_total for p in lora_totals), "Task 4: LoRA must use fewer params"
assert pct_of_full[0] < 1.0, "r=2 should be <1% of full"
print("✓ Checkpoint 3 passed — rank sweep visualised\n")



## TASK 5 — APPLY: Singapore law-firm assistant — rank selection


In [ ]:
# SCENARIO (illustrative): A Singapore law firm wants to adapt a 7B open-source base
# model to draft first-pass contracts in Singlish-aware business English,
# cite local statutes correctly, and handle the firm's preferred clause
# boilerplate.  They have 800 historical contracts for training and a
# single 24 GB GPU.
#
# DECISION: Full fine-tuning of a 7B model with Adam needs ~16 bytes per
# parameter (weights + gradients + two optimiser moments) ≈ 112 GB ->
# impossible on one 24 GB card.  LoRA is the only
# path.  The question is: what rank?
#
# RANK SELECTION GUIDE:
#   r=4:   <1% params — classification, light stylistic tweaks
#   r=8:   ~1% params — the pragmatic default
#   r=16:  ~2% params — complex domains (legal, medical, financial)
#   r=32+: diminishing returns; consider full FT if budget allows
#
# BUSINESS IMPACT (illustrative figures): junior associates currently spend ~6
# hours/week drafting first-pass contracts at a fully-loaded cost of
# ~S$120/hour (S$720/week per associate, 12 associates = S$8,640/week).
# A LoRA r=16 assistant trained overnight on the 800 contracts reduces
# that to ~1.5 hours/week of review-only work, saving ~S$6,480/week or
# ~S$324k/year over 50 working weeks.  Cloud GPU training cost: ~S$80 per run.
#
# RISK: a rank that is too low (r=2) underfits the firm's house style;
# a rank that is too high (r=128) starts to memorise individual contracts
# and leak client data in generation.  r=8 or r=16 is the sweet spot.

# Cost/benefit sweep for the law firm
print("Singapore law-firm rank cost/benefit:")
weekly_hours_before = 6 * 12
weekly_hours_after = 1.5 * 12
hourly_cost_sgd = 120
weekly_saving_sgd = (weekly_hours_before - weekly_hours_after) * hourly_cost_sgd
annual_saving_sgd = weekly_saving_sgd * 50  # 50 working weeks
print(f"  Hours/week before:  {weekly_hours_before}")
print(f"  Hours/week after:   {weekly_hours_after}")
print(f"  Weekly saving:      S${weekly_saving_sgd:,.0f}")
print(f"  Annual saving:      S${annual_saving_sgd:,.0f}")
print(f"  Training cost:      ~S$80 (one-off GPU run)")
print(
    f"  Recommended rank:   r=16 "
    f"({pct_of_full[ranks.index(16)]:.2f}% of full params)"
)



### Checkpoint 4


In [ ]:
assert annual_saving_sgd > 0, "Task 5: law firm should see positive savings"
print("✓ Checkpoint 4 passed — Singapore law firm cost/benefit analysed\n")



## REFLECTION


[x] Loaded an SFT dataset of instruction/response pairs (IMDB)
  [x] Built LoRALayer and LoRALinear from scratch (Kaiming A, zero B)
  [x] Verified identity-at-init: LoRA starts as W_new = W
  [x] Visualised the parameter reduction curve across ranks 2..128
  [x] Applied LoRA rank selection to a Singapore law-firm scenario
      (illustrative ~S$324k/year saving at r=16, ~S$80 training cost)

  KEY INSIGHT: LoRA is SVD applied to the UPDATE, not the weight.
  A handful of "directions" in weight space is usually enough to
  adapt a model to a new task, so 2*d*r parameters replace d^2.

  Next: 02_adapter_from_scratch.py builds bottleneck adapters and
  compares them to LoRA across four dimensions.


In [ ]:
print("=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

